In [1]:
from data import build_dataset
import polars as pl
from models.lightgcn import LightGCNModel

ds = build_dataset(split="leave_last_n", k=10, lln_n=10)
model = LightGCNModel.load("experiments/results/lightgcn_model_100epoch.pkl")

In [ ]:
from pathlib import Path

links = pl.read_csv(Path("data/ml-32m/links.csv"))
print(links.head())
print(links.schema)

# imdbId in links.csv is stored as integer without the 'tt' prefix
# confirm movieId in links matches keys in item2idx
sample_movie_ids = list(ds.item2idx.keys())[:5]
print(sample_movie_ids)
print(links.filter(pl.col("movieId").is_in(sample_movie_ids)))

shape: (5, 3)
┌─────────┬────────┬────────┐
│ movieId ┆ imdbId ┆ tmdbId │
│ ---     ┆ ---    ┆ ---    │
│ i64     ┆ i64    ┆ i64    │
╞═════════╪════════╪════════╡
│ 1       ┆ 114709 ┆ 862    │
│ 2       ┆ 113497 ┆ 8844   │
│ 3       ┆ 113228 ┆ 15602  │
│ 4       ┆ 114885 ┆ 31357  │
│ 5       ┆ 113041 ┆ 11862  │
└─────────┴────────┴────────┘
Schema({'movieId': Int64, 'imdbId': Int64, 'tmdbId': Int64})
[1, 2, 3, 4, 5]
shape: (5, 3)
┌─────────┬────────┬────────┐
│ movieId ┆ imdbId ┆ tmdbId │
│ ---     ┆ ---    ┆ ---    │
│ i64     ┆ i64    ┆ i64    │
╞═════════╪════════╪════════╡
│ 1       ┆ 114709 ┆ 862    │
│ 2       ┆ 113497 ┆ 8844   │
│ 3       ┆ 113228 ┆ 15602  │
│ 4       ┆ 114885 ┆ 31357  │
│ 5       ┆ 113041 ┆ 11862  │
└─────────┴────────┴────────┘


In [3]:
# Simulate a user who liked The Shawshank Redemption (tt0111161) and 
# The Godfather (tt0068646)
test_imdb_ids = [111161, 68646]  # tt prefix stripped, cast to int

# Step 1: imdbId → movieId
matched = links.filter(pl.col("imdbId").is_in(test_imdb_ids))
print(matched)

# Step 2: movieId → item_idx
movie_ids = matched["movieId"].to_list()
item_indices = [ds.item2idx.get(mid) for mid in movie_ids]
print(f"movieIds: {movie_ids}")
print(f"item_idxs: {item_indices}")

# Step 3: item_idx → embedding → proxy user vector
import numpy as np
valid = [i for i in item_indices if i is not None]
proxy_user = model._item_factors[valid].mean(axis=0)
print(f"proxy_user shape: {proxy_user.shape}")

# Step 4: score all items
scores = model._item_factors @ proxy_user
scores[valid] = -np.inf  # mask input items
top10 = np.argsort(-scores)[:10]

# Step 5: idx → movieId → title
top_movie_ids = [ds.idx2item[i] for i in top10]
top_titles = ds.movies_df.filter(pl.col("movieId").is_in(top_movie_ids))
print(top_titles)

shape: (2, 3)
┌─────────┬────────┬────────┐
│ movieId ┆ imdbId ┆ tmdbId │
│ ---     ┆ ---    ┆ ---    │
│ i64     ┆ i64    ┆ i64    │
╞═════════╪════════╪════════╡
│ 318     ┆ 111161 ┆ 278    │
│ 858     ┆ 68646  ┆ 238    │
└─────────┴────────┴────────┘
movieIds: [318, 858]
item_idxs: [314, 839]
proxy_user shape: (128,)
shape: (10, 3)
┌─────────┬─────────────────────────────────┬────────────────────────────────┐
│ movieId ┆ title                           ┆ genres                         │
│ ---     ┆ ---                             ┆ ---                            │
│ i32     ┆ str                             ┆ str                            │
╞═════════╪═════════════════════════════════╪════════════════════════════════╡
│ 50      ┆ Usual Suspects, The (1995)      ┆ Crime|Mystery|Thriller         │
│ 260     ┆ Star Wars: Episode IV - A New … ┆ Action|Adventure|Sci-Fi        │
│ 296     ┆ Pulp Fiction (1994)             ┆ Comedy|Crime|Drama|Thriller    │
│ 356     ┆ Forrest Gump (1994)

In [4]:
from inference import Recommender

rec = Recommender(
    model_path="experiments/results/lightgcn_model_100epoch.pkl",
    artifacts_path="experiments/results/inference_artifacts.pkl",
)

# Test 1: from IMDb IDs (Shawshank + Godfather)
print("=== Test 1: IMDb IDs ===")
print(rec.from_imdb_ids([111161, 68646], k=10))

# Test 2: from titles
print("\n=== Test 2: Title search ===")
print(rec.from_titles(["Shawshank Redemption", "The Godfather"], k=10))

# Test 3: title search with typos/partial names
print("\n=== Test 3: Fuzzy title search ===")
print(rec.from_titles(["shawshank", "godfather", "dark knight"], k=10))

# Test 4: edge case — unknown title
print("\n=== Test 4: Unknown title ===")
print(rec.from_titles(["xyznmojnkgjvljknlqds"], k=10))

=== Test 1: IMDb IDs ===
shape: (10, 4)
┌──────┬──────────────────────────────┬──────────────────────────────┬─────────────────────────────┐
│ rank ┆ title                        ┆ genres                       ┆ imdb_url                    │
│ ---  ┆ ---                          ┆ ---                          ┆ ---                         │
│ i64  ┆ str                          ┆ str                          ┆ str                         │
╞══════╪══════════════════════════════╪══════════════════════════════╪═════════════════════════════╡
│ 1    ┆ Pulp Fiction (1994)          ┆ Comedy|Crime|Drama|Thriller  ┆ https://www.imdb.com/title/ │
│      ┆                              ┆                              ┆ tt0…                        │
│ 2    ┆ Silence of the Lambs, The    ┆ Crime|Horror|Thriller        ┆ https://www.imdb.com/title/ │
│      ┆ (199…                        ┆                              ┆ tt0…                        │
│ 3    ┆ Matrix, The (1999)           ┆ Action|Sci-

In [5]:
print(ds.item2idx.get(858))  # should be non-None if it survived filtering
print(ds.train_df.filter(pl.col("item_idx") == ds.item2idx.get(858, -1)).shape)

839
(55583, 7)


In [6]:
from rapidfuzz import fuzz

titles = rec.movies["title"].to_list()
test_queries = ["shawshank", "godfather", "dark knight"]

for q in test_queries:
    from rapidfuzz import process
    matches = process.extract(q, titles, scorer=fuzz.WRatio, limit=5)
    print(f"\n'{q}':")
    for m in matches:
        print(f"  {m[1]:.1f}  {m[0]}")


'shawshank':
  80.0  Shawshank Redemption, The (1994)
  64.1  V. I. Warshawski (1991)
  63.5  Kashtanka (1952)
  63.5  Gharshana (2004)
  61.1  Ratsasan (2018)

'godfather':
  80.5  Disco Godfather (1979)
  80.0  Godfather, The (1972)
  80.0  Godfather: Part II, The (1974)
  80.0  Godfather: Part III, The (1990)
  80.0  Tokyo Godfathers (2003)

'dark knight':
  73.6  Dark Knight, The (2008)
  73.6  Dark Knight Rises, The (2012)
  73.6  The Dark Knight (2011)
  73.6  The Fire Rises: The Creation and Impact of The Dark Knight Trilogy (2013)
  73.6  Batman: The Dark Knight Returns (2013)


In [7]:
for q in test_queries:
    matches = process.extract(q, titles, scorer=fuzz.token_set_ratio, limit=5)
    print(f"\n'{q}':")
    for m in matches:
        print(f"  {m[1]:.1f}  {m[0]}")


'shawshank':
  50.0  Kshanam
  48.0  Kashtanka (1952)
  48.0  Gharshana (2004)
  46.7  Kshana Kshanam (1991)
  45.5  Tashan (2008)

'godfather':
  57.1  3 Godfathers (1948)
  57.1  After
  56.0  GodFather (1991)
  56.0  GodFather (2022)
  55.2  Our Godfather (2019)

'dark knight':
  60.0  3rd Night
  57.1  Dark Night (2018)
  57.1  Dark Light (2019)
  54.5  The Dark Knight (2011)
  54.5  Long Dark Night (2004)


In [8]:
import re

def normalize(title):
    title = title.lower().strip()
    title = re.sub(r'\(\d{4}\)', '', title).strip()
    title = re.sub(r'^(the|a|an)\s+', '', title).strip()
    title = re.sub(r',\s*(the|a|an)$', '', title).strip()
    return title

normalized_titles = [normalize(t) for t in titles]

for q in ["shawshank", "godfather", "dark knight"]:
    matches = process.extract(
        normalize(q), normalized_titles, scorer=fuzz.WRatio, limit=5
    )
    print(f"\n'{q}':")
    for m in matches:
        print(f"  {m[1]:.1f}  {titles[normalized_titles.index(m[0])]}")


'shawshank':
  90.0  Shawshank Redemption, The (1994)
  90.0  Shank (2010)
  90.0  Shank (2010)
  81.8  Shanks (1974)
  77.1  Shag (1989)

'godfather':
  100.0  Godfather, The (1972)
  100.0  Godfather, The (1972)
  100.0  Godfather, The (1972)
  95.0  Our Godfather (2019)
  90.0  Godfather: Part II, The (1974)

'dark knight':
  100.0  Dark Knight, The (2008)
  100.0  Dark Knight, The (2008)
  95.2  Dark Night (2018)
  90.0  Dark, The (2005)
  90.0  Dark Knight Rises, The (2012)


In [9]:
from rapidfuzz import process, fuzz
import re

def normalize(title):
    title = title.lower().strip()
    title = re.sub(r'\(\d{4}\)', '', title).strip()
    title = re.sub(r'^(the|a|an)\s+', '', title).strip()
    title = re.sub(r',\s*(the|a|an)$', '', title).strip()
    return title

titles = rec.movies["title"].to_list()
normalized_titles = [normalize(t) for t in titles]

matches = process.extract(
    normalize("xyzzy movie that does not exist"),
    normalized_titles,
    scorer=fuzz.WRatio,
    limit=3
)
for m in matches:
    print(f"  {m[1]:.1f}  {titles[normalized_titles.index(m[0])]}")

  90.0  Movie, A (1958)
  85.5  Tie That Binds, The (1995)
  85.5  Goofy Movie, A (1995)


In [10]:
results = rec.from_imdb_ids([111161, 68646], k=10)
print(results["imdb_url"].to_list())

['https://www.imdb.com/title/tt0110912', 'https://www.imdb.com/title/tt0102926', 'https://www.imdb.com/title/tt0133093', 'https://www.imdb.com/title/tt0109830', 'https://www.imdb.com/title/tt0108052', 'https://www.imdb.com/title/tt0137523', 'https://www.imdb.com/title/tt0114814', 'https://www.imdb.com/title/tt0076759', 'https://www.imdb.com/title/tt0167260', 'https://www.imdb.com/title/tt0080684']


In [11]:
print(rec.from_imdb_csv("user_exports/imdb_ranking_test_user.csv", min_rating=6.0, k=20))

shape: (20, 4)
┌──────┬──────────────────────────────┬──────────────────────────────┬─────────────────────────────┐
│ rank ┆ title                        ┆ genres                       ┆ imdb_url                    │
│ ---  ┆ ---                          ┆ ---                          ┆ ---                         │
│ i64  ┆ str                          ┆ str                          ┆ str                         │
╞══════╪══════════════════════════════╪══════════════════════════════╪═════════════════════════════╡
│ 1    ┆ Forrest Gump (1994)          ┆ Comedy|Drama|Romance|War     ┆ https://www.imdb.com/title/ │
│      ┆                              ┆                              ┆ tt0…                        │
│ 2    ┆ Matrix, The (1999)           ┆ Action|Sci-Fi|Thriller       ┆ https://www.imdb.com/title/ │
│      ┆                              ┆                              ┆ tt0…                        │
│ 3    ┆ Pulp Fiction (1994)          ┆ Comedy|Crime|Drama|Thriller  ┆ https

In [12]:
with pl.Config(
    fmt_str_lengths=100,
    tbl_width_chars=200,
    tbl_rows=20,
):
    print(rec.from_imdb_csv("user_exports/imdb_ranking_test_user.csv", min_rating=7.0, k=20))

shape: (20, 4)
┌──────┬────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────┬──────────────────────────────────────┐
│ rank ┆ title                                                                          ┆ genres                                          ┆ imdb_url                             │
│ ---  ┆ ---                                                                            ┆ ---                                             ┆ ---                                  │
│ i64  ┆ str                                                                            ┆ str                                             ┆ str                                  │
╞══════╪════════════════════════════════════════════════════════════════════════════════╪═════════════════════════════════════════════════╪══════════════════════════════════════╡
│ 1    ┆ Forrest Gump (1994)                                                            ┆ 

In [13]:
with pl.Config(
    fmt_str_lengths=100,
    tbl_width_chars=200,
    tbl_rows=20,
):
    print(rec.from_imdb_csv("user_exports/imdb_ranking_test_user.csv", min_rating=7.0, k=10))

shape: (10, 4)
┌──────┬───────────────────────────────────────────────────────┬─────────────────────────────────────────────────┬──────────────────────────────────────┐
│ rank ┆ title                                                 ┆ genres                                          ┆ imdb_url                             │
│ ---  ┆ ---                                                   ┆ ---                                             ┆ ---                                  │
│ i64  ┆ str                                                   ┆ str                                             ┆ str                                  │
╞══════╪═══════════════════════════════════════════════════════╪═════════════════════════════════════════════════╪══════════════════════════════════════╡
│ 1    ┆ Forrest Gump (1994)                                   ┆ Comedy|Drama|Romance|War                        ┆ https://www.imdb.com/title/tt0109830 │
│ 2    ┆ Matrix, The (1999)                                  

In [14]:
with pl.Config(
    fmt_str_lengths=100,
    tbl_width_chars=200,
    tbl_rows=20,
):
    # Test 1: from_imdb_ids standard
    print("=== Test 1: from_imdb_ids standard ===")
    print(rec.from_imdb_ids([111161, 68646], k=10))

    # Test 2: from_imdb_ids wildcard
    print("\n=== Test 2: from_imdb_ids wildcard ===")
    print(rec.from_imdb_ids([111161, 68646], k=10, wildcard=True, pool_size=75))

    # Test 3: wildcard is random — run twice, results should differ
    print("\n=== Test 3: wildcard randomness check ===")
    r1 = rec.from_imdb_ids([111161, 68646], k=10, wildcard=True)
    r2 = rec.from_imdb_ids([111161, 68646], k=10, wildcard=True)
    print(f"  Results differ: {r1['title'].to_list() != r2['title'].to_list()}")

    # Test 4: from_titles standard
    print("\n=== Test 4: from_titles standard ===")
    print(rec.from_titles(["Shawshank Redemption", "The Godfather"], k=10))

    # Test 5: from_titles wildcard
    print("\n=== Test 5: from_titles wildcard ===")
    print(rec.from_titles(["Shawshank Redemption", "The Godfather"], k=10, wildcard=True))

    # Test 6: from_imdb_csv standard
    print("\n=== Test 6: from_imdb_csv standard ===")
    print(rec.from_imdb_csv("user_exports/imdb_ranking_test_user.csv", min_rating=6.0, k=10))

    # Test 7: from_imdb_csv wildcard
    print("\n=== Test 7: from_imdb_csv wildcard ===")
    print(rec.from_imdb_csv("user_exports/imdb_ranking_test_user.csv", min_rating=6.0, k=10, wildcard=True))

    # Test 8: empty input
    print("\n=== Test 8: empty input ===")
    print(rec.from_titles(["xyzzy film that does not exist"], k=10))

    # Test 9: weighted vs unweighted differ on csv
    print("\n=== Test 9: weighted vs unweighted differ ===")
    r_weighted = rec.from_imdb_csv("user_exports/imdb_ranking_test_user.csv", min_rating=6.0, k=10)
    r_unweighted = rec.from_imdb_ids(
        [111161, 68646], k=10  # unweighted baseline
    )
    print(f"  Weighted top 3:   {r_weighted['title'].head(3).to_list()}")
    print(f"  Unweighted top 3: {r_unweighted['title'].head(3).to_list()}")

=== Test 1: from_imdb_ids standard ===
shape: (10, 4)
┌──────┬───────────────────────────────────────────────────────┬────────────────────────────────┬──────────────────────────────────────┐
│ rank ┆ title                                                 ┆ genres                         ┆ imdb_url                             │
│ ---  ┆ ---                                                   ┆ ---                            ┆ ---                                  │
│ i64  ┆ str                                                   ┆ str                            ┆ str                                  │
╞══════╪═══════════════════════════════════════════════════════╪════════════════════════════════╪══════════════════════════════════════╡
│ 1    ┆ Pulp Fiction (1994)                                   ┆ Comedy|Crime|Drama|Thriller    ┆ https://www.imdb.com/title/tt0110912 │
│ 2    ┆ Silence of the Lambs, The (1991)                      ┆ Crime|Horror|Thriller          ┆ https://www.imdb.com/title